# Урок 26 — Дерева та алгоритми дерев

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Перестаньте думати про дані як про плоску лінію. Дерева — це перехід від одновимірних структур (масив, список) до **багатовимірної ієрархічної організації пам'яті**.

> **Головна ідея:** Масиви чудові для лінійного зберігання, але коли даних стає багато — пошук займає $O(n)$. Дерева об'єднують гнучкість зв'язних списків із математичною ефективністю бінарного пошуку $O(\log n)$.

| Частина | Тема |
|---------|------|
| 1 | Чому існують дерева? Від плоских ліній до ієрархій |
| 2 | Термінологія та архітектура вузла |
| 3 | Бінарне дерево пошуку (BST): інваріант та пошук |
| 4 | Обхід у глибину (DFS): Pre-order, In-order, Post-order |
| 5 | Ітеративний DFS: явний стек замість рекурсії |
| 6 | Обхід у ширину (BFS): рівень за рівнем з чергою |
| 7 | Рекурсія проти Ітерації: компроміси |
| 8 | Збалансовані vs Вироджені дерева: складність $O(h)$ |
| 9 | BST проти Хеш-таблиці: архітектурний вибір |
| 10 | Питання для співбесіди |
| 11 | Підсумок |

---

---

## Частина 1: Чому існують дерева?

---

### Проблема одновимірних структур

Масиви та зв'язні списки — це **1D-структури**: елемент A стоїть за елементом B, і так до кінця.

```
Масив:       [1] → [2] → [3] → [4] → [5]   ← пошук O(n)
Дерево:           [4]
                 /   \
              [2]     [6]          ← пошук O(log n)
             / \     / \
           [1] [3] [5] [7]
```

**Ключовий компроміс:**
- Масив: швидкий доступ за індексом $O(1)$, але повільний пошук $O(n)$
- Зв'язний список: гнучка вставка $O(1)$, але пошук $O(n)$
- **Дерево:** поєднує гнучкість списку з ефективністю бінарного пошуку → $O(\log n)$

### Архітектурна природа дерева: Рекурсія та Divide & Conquer

Дерево — це **рекурсивна структура даних**. Кожна дитина вузла є коренем власного піддерева. Це робить дерева ідеальним двигуном для рекурсивних алгоритмів та парадигми "поділяй і володарюй".

```
Дерево = порожньо  АБО  (корінь + ліве_піддерево + праве_піддерево)
                                    ↑                    ↑
                             теж дерево!          теж дерево!
```

### Де ви використовуєте дерева щодня

| Система | Тип дерева | Навіщо |
|---------|-----------|--------|
| Файлова система | Загальне дерево | Папки містять файли та підпапки |
| HTML DOM | Дерево документа | `<html>` → `<body>` → `<div>` → ... |
| Бази даних | B-tree / B+-tree | Індексація для $O(\log n)$ запитів |
| Компілятори | AST (абстрактне синтаксичне дерево) | Розбір виразів: `2 + 3 * 4` |
| Автодоповнення | Trie (префіксне дерево) | Пошук за префіксом |

---

---

## Частина 2: Термінологія та архітектура вузла

---

### Словник термінів

Дерева традиційно малюють **перевернутими**: корінь зверху, листя знизу.

```
            [4]         ← Корінь (Root): єдина точка входу
           /   \
         [2]   [6]      ← Внутрішні вузли (Internal nodes)
        / \   / \
      [1] [3][5] [7]    ← Листки (Leaves): нащадків немає

Рівень 0:   [4]
Рівень 1:   [2], [6]
Рівень 2:   [1], [3], [5], [7]
```

| Термін | Визначення |
|--------|------------|
| **Вузол (Node)** | Контейнер з даними та вказівниками на нащадків |
| **Корінь (Root)** | Найвищий вузол, єдина точка входу |
| **Батько / Дитина (Parent / Child)** | Вузол, що посилається / на якого посилаються |
| **Листок (Leaf)** | Вузол без нащадків (`left=None`, `right=None`) |
| **Глибина (Depth)** | Кількість ребер від кореня до вузла |
| **Висота (Height)** | Максимальна глибина — довжина найдовшого шляху від кореня до листка |
| **Піддерево (Subtree)** | Вузол разом з усіма його нащадками |

> **Чому висота критична?** Складність більшості операцій з деревом — **$O(h)$**, де $h$ — висота. Якщо $h = \log n$ (збалансоване дерево) → $O(\log n)$. Якщо $h = n$ (вироджене) → $O(n)$.

---

In [ ]:
# ── Базова архітектура вузла бінарного дерева ─────────────────────────────────

class Node:
    """Вузол бінарного дерева: контейнер з даними та двома вказівниками."""

    def __init__(self, value):
        self.value = value    # корисні дані (ключ)
        self.left = None      # вказівник на ліве піддерево (менші значення)
        self.right = None     # вказівник на праве піддерево (більші значення)

    def __repr__(self):
        return f"Node({self.value})"


# ── Побудова дерева вручну для демонстрації ───────────────────────────────────
#
#         4
#        / \
#       2   6
#      / \ / \
#     1  3 5  7

root = Node(4)
root.left  = Node(2)
root.right = Node(6)
root.left.left   = Node(1)
root.left.right  = Node(3)
root.right.left  = Node(5)
root.right.right = Node(7)


# ── Допоміжні функції для аналізу дерева ─────────────────────────────────────

def height(node) -> int:
    """Рекурсивно обчислює висоту дерева. O(n) — відвідуємо кожен вузол."""
    if node is None:
        return -1  # висота порожнього дерева = -1 (або 0, залежно від конвенції)
    return 1 + max(height(node.left), height(node.right))


def count_nodes(node) -> int:
    """Рекурсивно підраховує кількість вузлів. O(n)."""
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


def is_leaf(node) -> bool:
    """Перевіряє чи є вузол листком (не має нащадків)."""
    return node.left is None and node.right is None


print("=== Аналіз нашого дерева ===")
print(f"  Корінь:         {root}")
print(f"  Висота:         {height(root)}")
print(f"  Кількість вузлів: {count_nodes(root)}")
print(f"  root — листок?  {is_leaf(root)}")
print(f"  Node(1) — листок? {is_leaf(root.left.left)}")
print()
print(f"  Глибина Node(3): 2 ребра від кореня (4 → 2 → 3)")
print(f"  Висота збалансованого дерева з 7 вузлів = log₂(7) ≈ {7 ** 0.5:.2f} → 2")

---

## Частина 3: Бінарне дерево пошуку (BST): інваріант та операції

---

### Інваріант BST

BST — це бінарне дерево з **жорстким глобальним правилом**:

> Для будь-якого вузла: **всі значення в лівому піддереві < значення вузла**, а **всі значення в правому піддереві > значення вузла**.

```
        4
       / \
      2   6        Правило виконується:
     / \ / \       1 < 2 < 3 < 4 < 5 < 6 < 7
    1  3 5  7
```

### Аналогія: Корпоративна ієрархія

Уявіть ієрархію компанії перевернутою донизу. Генеральний директор (корінь) має двох підлеглих: всі з ідентифікатором **менше** — у лівому відділі, **більше** — у правому. Кожен менеджер дотримується цього ж правила для своїх підлеглих.

### Типова пастка на співбесіді: локальний vs глобальний інваріант

❌ **Помилка джуніора:** перевірити лише чи поточний вузол більший за лівого і менший за правого.

✅ **Правильно:** інваріант має виконуватись для **всього піддерева**. Потрібно передавати дозволений діапазон `[min_val, max_val]` під час рекурсії:

```
       10
      /  \
     5    15    ← Виглядає правильно?
    / \
   3   12       ← 12 > 10! BST порушено!
                  (12 у лівому піддереві 10 — неприпустимо)
```

---

In [ ]:
# ── Операції BST: пошук, вставка, валідація ───────────────────────────────────

def bst_search(node, target):
    """
    Рекурсивний пошук у BST.
    Складність: O(h), де h — висота дерева.
    Збалансоване дерево: O(log n). Вироджене: O(n).
    """
    if node is None:             # базовий випадок: дійшли до порожнечі
        return False
    if node.value == target:     # знайдено!
        return True
    if target < node.value:      # шукаємо ліворуч — відкидаємо праву половину
        return bst_search(node.left, target)
    return bst_search(node.right, target)   # шукаємо праворуч


def bst_insert(node, value):
    """
    Рекурсивна вставка у BST.
    Повертає новий (або змінений) вузол — завжди присвоюйте результат!
    """
    if node is None:             # знайшли місце для вставки
        return Node(value)
    if value < node.value:       # нове значення менше → йдемо вліво
        node.left = bst_insert(node.left, value)
    elif value > node.value:     # більше → йдемо вправо
        node.right = bst_insert(node.right, value)
    # якщо value == node.value → дублікат, ігноруємо (стандартна поведінка)
    return node


def is_valid_bst(node, min_val=float('-inf'), max_val=float('inf')) -> bool:
    """
    Валідація BST за O(n) через передачу діапазону [min_val, max_val].
    Ключ: рухаємось вліво → поточний вузол стає новим max_val.
           рухаємось вправо → поточний вузол стає новим min_val.
    """
    if node is None:
        return True   # порожнє дерево — валідне BST
    if not (min_val < node.value < max_val):
        return False  # порушення глобального інваріанту!
    return (
        is_valid_bst(node.left,  min_val, node.value) and   # ліво: все менше за node
        is_valid_bst(node.right, node.value, max_val)       # право: все більше за node
    )


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Пошук у BST ===")
for val in [1, 5, 7, 9]:
    found = bst_search(root, val)
    print(f"  bst_search({val}) → {'знайдено ✓' if found else 'відсутній ✗'}")

print()
print("=== Вставка у BST ===")
# Будуємо нове BST з нуля
bst = None
for v in [5, 3, 7, 1, 4, 6, 8]:
    bst = bst_insert(bst, v)
    print(f"  Вставили {v}")
print(f"  Корінь: {bst}")
print(f"  Валідне BST: {is_valid_bst(bst)}")

print()
print("=== Валідація: невалідне BST ===")
# Створимо "зламане" дерево де 12 стоїть ліворуч від 10
broken = Node(10)
broken.left = Node(5)
broken.left.right = Node(12)  # 12 > 10 але в лівому піддереві — ПОМИЛКА!
broken.right = Node(15)
print(f"  is_valid_bst(broken) → {is_valid_bst(broken)}  (очікується False)") 

---

## Частина 4: Обхід у глибину (DFS) — три варіанти

---

### Аналогія: Дослідження лабіринту

DFS — це стратегія **«руки на стіні»**: ви йдете одним шляхом до самого кінця (глухого кута), тільки тоді повертаєтеся і пробуєте інший. DFS занурюється максимально глибоко перш ніж робити backtracking.

**Як DFS реалізується на рівні пам'яті?** Через **Стек (Stack - LIFO)**. Рекурсивний DFS неявно використовує Call Stack операційної системи: кожен виклик функції «заморожується» у стеку, поки не завершиться рекурсивний виклик нижче.

### Три варіанти DFS (різниця лише в позиції обробки вузла)

```
Дерево:
        4
       / \
      2   6
     / \ / \
    1  3 5  7

Pre-order   (Корінь → Лівий → Правий):  4, 2, 1, 3, 6, 5, 7
In-order    (Лівий → Корінь → Правий):  1, 2, 3, 4, 5, 6, 7  ← відсортований!
Post-order  (Лівий → Правий → Корінь):  1, 3, 2, 5, 7, 6, 4
```

| Тип обходу | Порядок | Застосування |
|------------|---------|-------------|
| **Pre-order** | Корінь, Лів, Прав | Копіювання дерева, серіалізація, префіксні вирази |
| **In-order** | Лів, Корінь, Прав | **Отримання відсортованих даних з BST** |
| **Post-order** | Лів, Прав, Корінь | Видалення дерева з пам'яті, обчислення розміру директорій |

> **In-order трюк для BST:** In-order обхід правильного BST завжди повертає елементи у зростаючому відсортованому порядку. Це пряме наслідування інваріанту BST.

> **Post-order і видалення:** Ви не можете видалити батька, поки не видалите дітей. Post-order гарантує, що спочатку обробляємо всіх нащадків.

---

In [ ]:
# ── Три варіанти DFS: рекурсивна реалізація ───────────────────────────────────

def preorder(node, result=None):
    """
    Pre-order: Корінь → Лівий → Правий.
    Обробляємо вузол ПЕРЕД тим, як заходимо в нащадків.
    Ідеально для копіювання структури дерева.
    """
    if result is None:
        result = []
    if node is not None:
        result.append(node.value)   # 1. спочатку обробляємо поточний вузол
        preorder(node.left,  result) # 2. потім занурюємось вліво
        preorder(node.right, result) # 3. потім вправо
    return result


def inorder(node, result=None):
    """
    In-order: Лівий → Корінь → Правий.
    Спочатку спускаємось максимально вліво, обробляємо, потім вправо.
    У BST: видає елементи у відсортованому за зростанням порядку.
    """
    if result is None:
        result = []
    if node is not None:
        inorder(node.left,  result)  # 1. максимально вглиб вліво
        result.append(node.value)    # 2. обробляємо поточний вузол
        inorder(node.right, result)  # 3. йдемо вправо
    return result


def postorder(node, result=None):
    """
    Post-order: Лівий → Правий → Корінь.
    Обробляємо батька ТІЛЬКИ після того, як обробили обох нащадків.
    Ідеально для видалення дерева або обчислення розміру директорій.
    """
    if result is None:
        result = []
    if node is not None:
        postorder(node.left,  result) # 1. спочатку всі нащадки зліва
        postorder(node.right, result) # 2. потім всі нащадки справа
        result.append(node.value)     # 3. тільки тоді обробляємо батька
    return result


# ── Демонстрація на нашому дереві ─────────────────────────────────────────────
print("=== Три варіанти DFS для дерева [4, 2, 6, 1, 3, 5, 7] ===")
print(f"  Pre-order  (Root→L→R):  {preorder(root)}")
print(f"  In-order   (L→Root→R):  {inorder(root)}   ← відсортовано!")
print(f"  Post-order (L→R→Root):  {postorder(root)}")
print()

# Практичний приклад: in-order як сортування
unsorted_data = [5, 2, 8, 1, 4, 7, 9]
tree = None
for v in unsorted_data:
    tree = bst_insert(tree, v)

sorted_via_tree = inorder(tree)
print(f"  Несортовані дані: {unsorted_data}")
print(f"  BST + In-order:   {sorted_via_tree}  ← Tree Sort!")
print(f"  Це еквівалентно sorted(): {sorted(unsorted_data)}")

---

## Частина 5: Ітеративний DFS — явний стек

---

### Проблема рекурсії у виробничому коді

Рекурсивний DFS — елегантний, але небезпечний. Python має ліміт стека викликів (~1000 рівнів). Вироджене дерево з 5000 вузлів → `RecursionError`.

**Рішення Senior-інженера:** замінити Call Stack на явний `list` як стек.

### Архітектурний трюк Pre-order ітеративного DFS

Щоб **лівий** вузол оброблявся першим (стояв на вершині стека), треба класти у стек спочатку **правого** нащадка, а потім лівого:

```
Крок 1: стек = [4]
Крок 2: витягуємо 4, кладемо right=6, потім left=2 → стек = [6, 2]
Крок 3: витягуємо 2 (LIFO!), кладемо right=3, left=1  → стек = [6, 3, 1]
Крок 4: витягуємо 1 (листок)                           → стек = [6, 3]
...і так далі
Порядок: 4, 2, 1, 3, 6, 5, 7  ← Pre-order!
```

---

In [ ]:
# ── Ітеративний DFS: Pre-order та In-order через явний стек ──────────────────

def iterative_preorder(root):
    """
    Ітеративний Pre-order DFS: Корінь → Лівий → Правий.
    Замість Call Stack використовуємо явний list як LIFO-стек.
    Ніколи не викликає RecursionError.
    """
    if root is None:
        return []

    result = []
    stack = [root]   # list як стек — append/pop працюють O(1)

    while stack:
        node = stack.pop()          # LIFO: знімаємо з вершини
        result.append(node.value)   # обробляємо вузол

        # КЛЮЧОВИЙ ТРЮК: правого кладемо ПЕРШИМ → лівий опиниться на вершині
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)

    return result


def iterative_inorder(root):
    """
    Ітеративний In-order DFS: Лівий → Корінь → Правий.
    Складніший — потрібно «занурюватись» вліво перед обробкою вузла.
    """
    result = []
    stack = []
    current = root

    while current is not None or stack:
        # Спускаємось максимально вліво, складаючи вузли у стек
        while current is not None:
            stack.append(current)   # «заморожуємо» вузол для майбутньої обробки
            current = current.left  # продовжуємо спуск вліво

        # Дійшли до найлівішого вузла — обробляємо
        current = stack.pop()
        result.append(current.value)  # обробка: лівий обхід завершено
        current = current.right       # тепер переходимо на правого нащадка

    return result


def dfs_search_iterative(root, target) -> bool:
    """Пошук значення через ітеративний DFS. Повертає True якщо знайдено."""
    if root is None:
        return False
    stack = [root]
    while stack:
        node = stack.pop()
        if node.value == target:
            return True
        if node.right: stack.append(node.right)  # правий першим
        if node.left:  stack.append(node.left)   # лівий на вершині
    return False


# ── Порівняння: рекурсивний vs ітеративний ────────────────────────────────────
print("=== Рекурсивний vs Ітеративний ===")
print(f"  Рекурсивний Pre-order:   {preorder(root)}")
print(f"  Ітеративний Pre-order:   {iterative_preorder(root)}")
print(f"  Результати однакові: {preorder(root) == iterative_preorder(root)}")
print()
print(f"  Рекурсивний In-order:    {inorder(root)}")
print(f"  Ітеративний In-order:    {iterative_inorder(root)}")
print(f"  Результати однакові: {inorder(root) == iterative_inorder(root)}")
print()
print("=== Демонстрація RecursionError на вирогженому дереві ===")

import sys
# Будуємо вироджене (праве) дерево глибиною 100
degenerate = Node(0)
cur = degenerate
for i in range(1, 100):
    cur.right = Node(i)
    cur = cur.right

# Ітеративний — безпечний
result = iterative_preorder(degenerate)
print(f"  Ітеративний DFS на дереві глибиною 100: {len(result)} вузлів — OK ✓")

# Рекурсивний — безпечний для 100, але при 5000 упаде
print(f"  Ліміт рекурсії Python: {sys.getrecursionlimit()} кадрів")
print(f"  Вироджене дерево з 5000 вузлів → RecursionError при рекурсивному обході!")

---

## Частина 6: Обхід у ширину (BFS) — рівень за рівнем

---

### Аналогія: Читання книги рядок за рядком

BFS читає дерево горизонтально: спочатку весь перший рядок (рівень 0: корінь), потім весь другий рядок (рівень 1), потім третій (рівень 2).

Уявіть камінь, кинутий у воду: кола розширюються рівномірно на всі боки — це BFS.

### Чому BFS потребує Черги (Queue, FIFO)?

DFS використовує **Стек (LIFO)** — останній доданий обробляється першим → глибина.
BFS використовує **Чергу (FIFO)** — перший доданий обробляється першим → ширина.

```
BFS покроково для [4, 2, 6, 1, 3, 5, 7]:

Старт:   Queue = [4]
Крок 1:  дістаємо 4  → додаємо 2, 6    → Queue = [2, 6]
Крок 2:  дістаємо 2  → додаємо 1, 3    → Queue = [6, 1, 3]
Крок 3:  дістаємо 6  → додаємо 5, 7    → Queue = [1, 3, 5, 7]
Кроки 4-7: вузли 1,3,5,7 — листки, нащадків немає
Порядок: 4, 2, 6, 1, 3, 5, 7  ← рівень за рівнем!
```

### Критична помилка Python: `list.pop(0)` vs `deque.popleft()`

```python
# ❌ НЕПРАВИЛЬНО: list.pop(0) — O(n)! Зсуває всі елементи у пам'яті.
queue = [root]
node = queue.pop(0)   # O(n) — весь алгоритм стає O(n²)!

# ✅ ПРАВИЛЬНО: deque.popleft() — O(1). Завжди так!
from collections import deque
queue = deque([root])
node = queue.popleft()  # O(1)
```

---

In [ ]:
# ── BFS: обхід рівень за рівнем ───────────────────────────────────────────────
from collections import deque

def bfs_traversal(root):
    """
    BFS / Level-order traversal.
    Складність: O(n) час, O(w) пам'ять де w — максимальна ширина дерева.
    У збалансованому дереві нижній рівень ≈ n/2 вузлів → O(n) пам'ять.
    """
    if root is None:
        return []

    result = []
    queue = deque([root])  # deque для O(1) операцій з обох боків

    while queue:
        node = queue.popleft()          # FIFO: беремо перший — O(1)!
        result.append(node.value)

        # Спочатку лівого, потім правого → зберігаємо порядок зліва направо
        if node.left:  queue.append(node.left)
        if node.right: queue.append(node.right)

    return result


def bfs_by_levels(root):
    """
    BFS з розбиттям по рівнях: повертає список списків.
    Корисно коли потрібно знати рівень кожного вузла.
    """
    if root is None:
        return []

    levels = []
    queue = deque([root])

    while queue:
        level_size = len(queue)   # скільки вузлів на поточному рівні
        current_level = []

        for _ in range(level_size):  # обробляємо рівно один рівень
            node = queue.popleft()
            current_level.append(node.value)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)

        levels.append(current_level)

    return levels


def bfs_search(root, target) -> bool:
    """Пошук значення через BFS. Гарантує знаходження найближчого по рівню."""
    if root is None:
        return False
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node.value == target:
            return True
        if node.left:  queue.append(node.left)
        if node.right: queue.append(node.right)
    return False


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== BFS обхід ===")
print(f"  Level-order:   {bfs_traversal(root)}")
print()
print("=== BFS по рівнях ===")
for i, level in enumerate(bfs_by_levels(root)):
    print(f"  Рівень {i}: {level}")
print()

print("=== Порівняння всіх обходів ===")
print(f"  Pre-order  (DFS): {preorder(root)}     ← глибина: корінь першим")
print(f"  In-order   (DFS): {inorder(root)}   ← глибина: відсортовано")
print(f"  Post-order (DFS): {postorder(root)}  ← глибина: корінь останнім")
print(f"  Level-order(BFS): {bfs_traversal(root)}  ← ширина: рівень за рівнем")
print()
print("=== Коли використовувати BFS? ===")
print("  ✓ Знаходження НАЙКОРОТШОГО шляху (найменша кількість кроків/рівнів)")
print("  ✓ Перевірка чи дерево є повним/ідеально збалансованим")
print("  ✓ Серіалізація дерева рівень за рівнем")
print("  ✓ Знаходження всіх вузлів на певній глибині")

---

## Частина 7: Рекурсія проти Ітерації — компроміси

---

### Як рекурсія працює на рівні пам'яті

Рекурсивний DFS **неявно** використовує Call Stack операційної системи. Кожен виклик функції:
1. «Заморожує» стан поточного вузла (локальні змінні, адресу повернення)
2. Кладе «кадр» у системний стек
3. Занурюється у вкладений виклик
4. При поверненні — «розморожує» кадр зі стека

```
inorder(4)                     ← кадр 1 у стеку
  → inorder(2)                 ← кадр 2
       → inorder(1)            ← кадр 3
            → inorder(None)    ← кадр 4: базовий випадок, повертаємось
       ← обробляємо 1
       → inorder(None)         ← кадр 4
  ← обробляємо 2
  → inorder(3) ...
← обробляємо 4
```

### Порівняльна таблиця

| Критерій | Рекурсія | Ітерація (явний стек/черга) |
|----------|----------|----------------------------|
| **Читабельність** | Дуже висока, елегантна | Більш детальна |
| **Безпека** | `RecursionError` при глибині > ~1000 | Безпечна для будь-якої глибини |
| **BFS** | Неможливий через рекурсію (системний стек ≠ черга) | Єдиний правильний спосіб |
| **Контроль** | Складно зупинити/призупинити | Повний контроль |
| **Пам'ять** | $O(h)$ на системному стеку | $O(h)$ на явному стеку |

> **Senior-інсайт:** Post-order ітеративний DFS — найскладніший для реалізації, бо вузол треба відвідати двічі: спочатку для маршрутизації до нащадків, потім для обробки після повернення з них. Потрібно відстежувати `last_visited` вузол.

---

In [ ]:
# ── Демонстрація: рекурсивна глибина стека ───────────────────────────────────
import sys
import traceback

print("=== Демонстрація: рекурсія vs ітерація ===")

# Будуємо праве вироджене дерево різних розмірів
def build_right_skewed(n):
    """Будує 'косе' праве дерево: 0 → 1 → 2 → ... → n-1."""
    if n == 0: return None
    head = Node(0)
    cur = head
    for i in range(1, n):
        cur.right = Node(i)
        cur = cur.right
    return head

# Тестуємо ітеративний — він завжди безпечний
for size in [100, 500, 900, 990]:
    tree = build_right_skewed(size)
    result = iterative_preorder(tree)
    print(f"  Ітеративний, глибина={size}: {len(result)} вузлів — OK ✓")

print()
# Рекурсивний при глибині близькій до ліміту
original_limit = sys.getrecursionlimit()
sys.setrecursionlimit(200)  # тимчасово зменшуємо для демонстрації
tree_deep = build_right_skewed(150)
try:
    preorder(tree_deep)  # має впасти
    print("  Рекурсивний, глибина=150: OK (ліміт не перевищено)")
except RecursionError:
    print(f"  Рекурсивний, глибина=150: RecursionError! ← при ліміті {sys.getrecursionlimit()} кадрів")
    print(f"  Ітеративний тієї ж глибини: {len(iterative_preorder(tree_deep))} вузлів — OK ✓")
finally:
    sys.setrecursionlimit(original_limit)  # відновлюємо оригінальний ліміт

print()
print(f"  Оригінальний ліміт відновлено: {sys.getrecursionlimit()}")

---

## Частина 8: Збалансовані vs Вироджені дерева — $O(h)$

---

### Трагедія відсортованих даних

Якщо вставити у порожнє BST вже відсортовані дані, кожен новий вузол піде виключно вправо. Дерево перетвориться на **«сосиску»** — виродиться у зв'язний список:

```
Вставка: 1, 2, 3, 4, 5

Збалансоване BST:          Вироджене BST:
        3                  1
       / \                  \
      1   4                  2
       \ / \                  \
        2  5                  3
                               \
         h = log(5) ≈ 2         4
         Пошук: O(log n)         \
                                  5
                              h = 5 = n
                              Пошук: O(n)
```

### Складність $O(h)$: залежність від висоти

| Тип дерева | Висота $h$ | Пошук/Вставка/Видалення |
|-----------|-----------|-------------------------|
| **Ідеально збалансоване** | $\log_2 n$ | $O(\log n)$ |
| **Типове BST** (випадкові дані) | $\approx 1.38 \log_2 n$ | $O(\log n)$ в середньому |
| **Вироджене** (відсортовані дані) | $n - 1$ | $O(n)$ |

### Рішення: Самобалансуючі дерева

У виробничих системах використовують **самобалансуючі дерева** — вони автоматично виконують «повороти» вузлів при вставці, гарантуючи $O(\log n)$:

- **AVL-дерева:** Жорстке балансування — різниця висот лівого і правого піддерев ≤ 1. Більше поворотів, але ідеальний баланс.
- **Червоно-чорні дерева:** М'якше балансування через кольорові мітки. Менше поворотів, трохи менш збалансовані. Використовується в `std::map` (C++), `TreeMap` (Java).
- **B-дерева / B+-дерева:** Кожен вузол має сотні нащадків. «Низьке і широке» дерево → мінімум зверненть до диску. Використовується в PostgreSQL, MySQL для індексів.

---

In [ ]:
# ── Демонстрація деградації BST при відсортованих даних ───────────────────────
import timeit

def bst_search_count(node, target):
    """Пошук з підрахунком кроків."""
    steps = 0
    while node:
        steps += 1
        if node.value == target:
            return steps
        elif target < node.value:
            node = node.left
        else:
            node = node.right
    return steps


print("=== Збалансоване BST vs Вироджене BST ===")
print()

N = 1000

# Вироджене BST (відсортована вставка)
skewed_bst = None
for v in range(1, N + 1):         # 1, 2, 3, ..., N → вироджується!
    skewed_bst = bst_insert(skewed_bst, v)

# Відносно збалансоване BST (випадкова вставка)
import random
random.seed(42)
balanced_bst = None
shuffled = list(range(1, N + 1))
random.shuffle(shuffled)
for v in shuffled:                # випадковий порядок → краще збалансовано
    balanced_bst = bst_insert(balanced_bst, v)

target = N  # найгірший випадок — шукаємо останній елемент

skewed_steps   = bst_search_count(skewed_bst,   target)
balanced_steps = bst_search_count(balanced_bst, target)

print(f"  n = {N} елементів")
print(f"  Вироджене BST:    висота ≈ {height(skewed_bst)},  кроків пошуку: {skewed_steps}")
print(f"  Збалансоване BST: висота ≈ {height(balanced_bst)},  кроків пошуку: {balanced_steps}")
print(f"  Теоретичний log₂({N}) ≈ {N.bit_length() - 1}")
print()
print(f"  Вироджене дерево повільніше у {skewed_steps // balanced_steps}x разів!")
print()
print("=== Висновок ===")
print(f"  O(log n): збалансоване BST з {N} вузлів → {balanced_steps} кроків")
print(f"  O(n):     вироджене BST з {N} вузлів    → {skewed_steps} кроків")
print(f"  Різниця при n={N}: {'%.0f' % (skewed_steps/balanced_steps)}x")

---

## Частина 9: BST проти Хеш-таблиці — архітектурний вибір

---

### Ключова відмінність: Порядок vs Швидкість

Хеш-функція **знищує** будь-яке поняття «близькості» між значеннями. Ключі `10` і `11` у хеш-таблиці можуть бути в абсолютно різних bucket'ах.

BST **зберігає** порядок: кожен крок по дереву — це навігаційне рішення на основі відношення «менше/більше».

| Операція | `dict`/`set` (Hash) | BST (збалансоване) |
|----------|---------------------|--------------------|
| Точний пошук | $O(1)$ середнє | $O(\log n)$ |
| Вставка | $O(1)$ середнє | $O(\log n)$ |
| Видалення | $O(1)$ середнє | $O(\log n)$ |
| Мінімум/Максимум | $O(n)$ — сканування | $O(\log n)$ — крайній лівий/правий |
| Діапазонний запит [a, b] | $O(n)$ — повний перебір | $O(\log n + k)$ |
| Відсортована ітерація | $O(n \log n)$ — сортування | $O(n)$ — in-order |
| Найближчий сусід | Неможливо без сканування | $O(\log n)$ |
| Пам'ять | ~3x більше (розріджений масив) | ~2x (вказівники) |

### Алгоритм вибору структури

```
Яка операція найважливіша?
         │
    ┌────┴─────────────────┐
    │                      │
Тільки точний пошук    Потрібен порядок?
("є чи немає")         (мін/макс, діапазон, сортування)
    │                      │
dict / set              BST (або SortedList від
O(1) пошук             бібліотеки sortedcontainers)
```

> **Важливо:** У стандартній бібліотеці Python **немає вбудованого BST**. Для впорядкованих динамічних колекцій використовують `sortedcontainers.SortedList` або `bintrees`. Базовий `dict` в Python 3.7+ зберігає порядок **вставки** (не сортований порядок!) — це різні речі.

---

In [ ]:
# ── BST vs dict: порівняння на діапазонних запитах ───────────────────────────
import timeit
import random

def bst_range_query(node, low, high, result=None):
    """
    Діапазонний запит: знайти всі значення у [low, high].
    BST: O(log n + k), де k — кількість знайдених елементів.
    Ключ: якщо node.value < low — відкидаємо ліве піддерево.
           якщо node.value > high — відкидаємо праве піддерево.
    """
    if result is None:
        result = []
    if node is None:
        return result
    if node.value > low:          # є кандидати у лівому піддереві
        bst_range_query(node.left, low, high, result)
    if low <= node.value <= high:  # поточний вузол у діапазоні
        result.append(node.value)
    if node.value < high:          # є кандидати у правому піддереві
        bst_range_query(node.right, low, high, result)
    return result


# Будуємо BST та dict з однакових даних
random.seed(42)
N = 10_000
data = random.sample(range(1, 100_001), N)   # N унікальних чисел від 1 до 100000

# BST
bst_tree = None
for v in data:
    bst_tree = bst_insert(bst_tree, v)

# dict
data_dict = {v: True for v in data}

LOW, HIGH = 1000, 2000

# Діапазонний запит через BST
bst_result = bst_range_query(bst_tree, LOW, HIGH)

# Діапазонний запит через dict — треба перебрати ВСІ ключі
dict_result = [k for k in data_dict if LOW <= k <= HIGH]

print(f"=== Діапазонний запит [{LOW}, {HIGH}] в колекції з {N} елементів ===")
print(f"  BST знайдено: {len(bst_result)} елементів")
print(f"  Dict знайдено: {len(dict_result)} елементів")
print(f"  Результати однакові: {sorted(bst_result) == sorted(dict_result)}")
print()

# Benchmark
t_bst = timeit.timeit(
    stmt='bst_range_query(bst_tree, 1000, 2000)',
    globals={'bst_range_query': bst_range_query, 'bst_tree': bst_tree},
    number=1000
) / 1000 * 1000

t_dict = timeit.timeit(
    stmt='[k for k in data_dict if 1000 <= k <= 2000]',
    globals={'data_dict': data_dict},
    number=1000
) / 1000 * 1000

print(f"  Швидкість BST range query: {t_bst:.3f} мс")
print(f"  Швидкість dict range query: {t_dict:.3f} мс")
print(f"  BST швидше для діапазонних запитів: {t_dict / t_bst:.1f}x")
print()
print("  Висновок: BST O(log n + k) >> dict O(n) для range queries!")

---

## Частина 10: Питання для співбесіди

---

### Питання 1: In-order обхід та відсортований масив

> **Запитання:** Дано BST. Поверніть всі значення у відсортованому порядку без сортування.

**Коротка відповідь:** In-order обхід BST (Лів → Корінь → Прав) завжди повертає елементи у відсортованому порядку завдяки інваріанту BST.

```python
def inorder(node, result=[]):
    if node:
        inorder(node.left, result)
        result.append(node.value)
        inorder(node.right, result)
    return result
```

**Типова помилка:** Використовувати `result=[]` як default argument — це mutable default argument trap: список буде спільним між усіма викликами! Правильно: `result=None` → `if result is None: result = []`.

---

### Питання 2: Чому BFS, а не DFS для найкоротшого шляху?

> **Запитання:** Дано дерево. Знайдіть мінімальну глибину (найкоротший шлях від кореня до листка). Чому слід використовувати BFS?

**Коротка відповідь:** BFS гарантовано знаходить **перший** листок на мінімальній глибині, бо обходить рівень за рівнем. DFS може піти по довгому шляху до глибокого листка перш ніж перевірить мілкий.

**Типова помилка:** Використовувати `list.pop(0)` замість `deque.popleft()` — перший є $O(n)$ і перетворює $O(n)$ BFS на $O(n^2)$.

---

### Питання 3: Валідація BST — локальний vs глобальний інваріант

> **Запитання:** Напишіть функцію `is_valid_bst(root)`. Яка найпоширеніша помилка?

**Коротка відповідь:** Недостатньо перевіряти лише `node.left.value < node.value < node.right.value`. Треба передавати діапазон `[min_val, max_val]` у кожен виклик.

```python
def is_valid_bst(node, min_val=float('-inf'), max_val=float('inf')):
    if not node: return True
    if not (min_val < node.value < max_val): return False
    return (is_valid_bst(node.left,  min_val, node.value) and
            is_valid_bst(node.right, node.value, max_val))
```

---

### Питання 4: Stack Overflow / RecursionError

> **Запитання:** Рекурсивний DFS на дереві з 1 000 000 вузлів. Яка проблема? Як вирішити?

**Коротка відповідь:** Вироджене дерево → глибина 1 000 000 → `RecursionError` (ліміт Python ~1000). Рішення: ітеративний DFS з явним `list` як стеком — не використовує системний стек.

---

### Питання 5: Pre-order, In-order, Post-order — де що застосовується?

> **Запитання:** Коли використовувати кожен з трьох видів DFS?

| Обхід | Застосування |
|-------|--------------|
| **Pre-order** | Копіювання дерева (батько вставляється до дітей), серіалізація |
| **In-order** | Отримання відсортованих даних з BST |
| **Post-order** | Видалення дерева (спочатку діти, потім батько), розмір директорій |

---

---

## Підсумок Уроку

---

| Концепція | Структура даних | Складність |
|-----------|----------------|------------|
| **BST пошук/вставка** | Дерево (збалансоване) | $O(\log n)$ |
| **BST (вироджене)** | Пов'язаний список | $O(n)$ |
| **DFS (всі варіанти)** | Стек (Call Stack або явний) | $O(n)$ час, $O(h)$ пам'ять |
| **BFS / Level-order** | Черга `deque` | $O(n)$ час, $O(w)$ пам'ять |
| **Висота дерева** | — | $O(n)$ для обчислення |

### Ключові архітектурні правила:

- **Висота $h$ = ключ до складності**: $O(h)$ = $O(\log n)$ для збалансованого, $O(n)$ для виродженого
- **DFS ↔ Стек (LIFO)**, **BFS ↔ Черга (FIFO)** — це архітектурний принцип, не деталь
- **In-order BST = відсортовані дані** — пряме наслідування інваріанту BST
- **Валідація BST** потребує передачі глобального діапазону `[min, max]`, не лише локальної перевірки
- **`deque.popleft()` = $O(1)$**, `list.pop(0)` = $O(n)$ — ніколи не плутати!
- **dict** перемагає для точного пошуку $O(1)$; **BST** перемагає для діапазонних запитів та сортованої ітерації

### Вибір алгоритму обходу:

```
Яка задача?
      │
  ┌───┴──────────────────────────────┐
  │                                  │
Рівневий обхід /              Глибинний обхід
Найкоротший шлях              (порядок обробки)
      │                              │
    BFS                    ┌─────────┼─────────┐
   deque                Pre-order  In-order Post-order
                        (копія)  (сортоване) (видалення)
```

> **Головний Senior-інсайт:**
> Дерева — це матеріалізація рекурсивного мислення. BST матеріалізує бінарний пошук. DFS матеріалізує стек. BFS матеріалізує чергу. Вибір структури даних — це вибір ментальної моделі.

---
*Наступний урок: Алгоритми сортування*